In [2]:
# %% [Cell 1] Environment, Global Paths & Dynamic Class Discovery

import os
import io
import gc
import json
import time
import shutil
import base64
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# Auto-install necessary dependencies in Kaggle
import subprocess, sys
for pkg in ["timm", "albumentations", "ultralytics", "pycocotools", "fastapi", "uvicorn", "python-multipart"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from ultralytics import YOLO
from pycocotools import mask as maskUtils

# ---------------------------------------------------------------------------
# 1.1 Reproducibility & Device Configuration
# ---------------------------------------------------------------------------
SEED = 42
def seed_everything(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[System] Active Hardware: {DEVICE} (Total GPUs: {torch.cuda.device_count()})")

# ---------------------------------------------------------------------------
# 1.2 Path Configuration & Background Filtering Rule
# ---------------------------------------------------------------------------
@dataclass
class MasterConfig:
    # Exact dataset path specified
    dataset_root: str = "/kaggle/input/datasets/sadidahmedturjo/native-vehicle-seg-and-det-dataset/vehicle_dataset_v0.1.2"
    output_dir: str = "/kaggle/working"
    
    # Subfolder locations
    train_dir: str = ""
    val_dir: str = ""
    test_dir: str = ""
    train_ann: str = ""
    val_ann: str = ""
    test_ann: str = ""
    
    # Destination directories
    yolo_export_dir: str = ""
    checkpoint_dir: str = ""
    results_dir: str = ""

    # Hardware & Training Hyperparameters
    img_size: int = 640
    batch_size: int = 16
    num_workers: int = 2
    pin_memory: bool = True
    mixed_precision: bool = True
    epochs: int = 50
    lr_backbone: float = 1e-5
    lr_aspp_ca: float = 5e-5
    lr_decoder: float = 1e-4
    early_stop_patience: int = 6
    
    # YOLO parameters
    yolo_model_variant: str = "yolo11s-seg.pt"
    yolo_epochs: int = 50

    # Class mappings
    class_names: List[str] = field(default_factory=list)
    category_id_to_label: Dict[int, int] = field(default_factory=dict)
    num_classes: int = 0

    def __post_init__(self):
        self.train_dir = os.path.join(self.dataset_root, "train")
        self.val_dir = os.path.join(self.dataset_root, "valid")
        self.test_dir = os.path.join(self.dataset_root, "test")
        
        self.train_ann = os.path.join(self.train_dir, "_annotations.coco.json")
        self.val_ann = os.path.join(self.val_dir, "_annotations.coco.json")
        self.test_ann = os.path.join(self.test_dir, "_annotations.coco.json")

        self.yolo_export_dir = os.path.join(self.output_dir, "yolo_dataset")
        self.checkpoint_dir = os.path.join(self.output_dir, "checkpoints")
        self.results_dir = os.path.join(self.output_dir, "thesis_results")

        os.makedirs(self.checkpoint_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(self.yolo_export_dir, exist_ok=True)

        # -------------------------------------------------------------------
        # Discard "object" / "objects" as background (Index 0)
        # -------------------------------------------------------------------
        IGNORE_CLASSES = {"object", "objects", "background", "none"}
        if os.path.exists(self.train_ann):
            with open(self.train_ann, "r") as f:
                coco = json.load(f)
            
            # Filter out generic 'object' class
            valid_cats = [c for c in coco["categories"] if c["name"].lower().strip() not in IGNORE_CLASSES]
            valid_cats = sorted(valid_cats, key=lambda c: c["id"])

            self.class_names = ["background"] + [c["name"] for c in valid_cats]
            # Valid classes get contiguous labels 1..K. Ignored classes default to 0 (background)
            self.category_id_to_label = {c["id"]: i + 1 for i, c in enumerate(valid_cats)}
            self.num_classes = len(self.class_names)
            print(f"[Config] Dynamically discovered {self.num_classes} classes (Class 'object' filtered to Background):")
            print(f"         {self.class_names}")
        else:
            self.class_names = ["background", "bus", "truck", "car", "leguna", "auto-rickshaw", "motorcycle"]
            self.category_id_to_label = {i: i for i in range(1, len(self.class_names))}
            self.num_classes = len(self.class_names)

cfg = MasterConfig()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
[System] Active Hardware: cuda (Total GPUs: 2)
[Config] Dynamically discovered 8 classes (Class 'object' filtered to Background):
         ['background', 'auto_rickshaw', 'bus', 'car', 'cng', 'leguna', 'motorcycle', 'truck']


In [3]:
# %% [Cell 2] COCO-to-YOLO Converter & Minority Instance Copy-Paste Augmentation
import os
import json
import shutil
import random
from pathlib import Path
from typing import Dict, Tuple

import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset, DataLoader
from pycocotools import mask as maskUtils
cv2.setNumThreads(0) 
# ---------------------------------------------------------------------------
# 2.1 COCO Mask Parser
# ---------------------------------------------------------------------------
def segmentation_to_binary_mask(segmentation, orig_h: int, orig_w: int) -> np.ndarray:
    if isinstance(segmentation, dict):
        rle = segmentation
        if isinstance(rle["counts"], list):
            rle = maskUtils.frPyObjects(rle, orig_h, orig_w)
        binary = maskUtils.decode(rle)
        if binary.ndim == 3:
            binary = np.any(binary, axis=2)
        return binary.astype(np.uint8)

    if isinstance(segmentation, list) and len(segmentation) > 0 and isinstance(segmentation[0], dict):
        binary = np.zeros((orig_h, orig_w), dtype=np.uint8)
        for rle in segmentation:
            if isinstance(rle["counts"], list):
                rle = maskUtils.frPyObjects(rle, orig_h, orig_w)
            part = maskUtils.decode(rle)
            if part.ndim == 3:
                part = np.any(part, axis=2)
            binary = np.maximum(binary, part.astype(np.uint8))
        return binary

    binary = np.zeros((orig_h, orig_w), dtype=np.uint8)
    for poly in segmentation:
        if len(poly) < 6:
            continue
        pts = np.array(poly, dtype=np.float32).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(binary, [pts], 1)
    return binary

# ---------------------------------------------------------------------------
# 2.2 Convert COCO Dataset to YOLO-Seg (Excluding "object")
# ---------------------------------------------------------------------------
def convert_coco_to_yolo(coco_ann_path: str, img_src_dir: str, yolo_split_dir: str, cat_to_idx: Dict[int, int]):
    if not os.path.exists(coco_ann_path):
        return
    images_dst = os.path.join(yolo_split_dir, "images")
    labels_dst = os.path.join(yolo_split_dir, "labels")
    os.makedirs(images_dst, exist_ok=True)
    os.makedirs(labels_dst, exist_ok=True)

    with open(coco_ann_path, "r") as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco.get("images", [])}
    anns_by_img = {}
    for ann in coco.get("annotations", []):
        anns_by_img.setdefault(ann["image_id"], []).append(ann)

    for img_id, img_info in images.items():
        fname = img_info["file_name"]
        w_orig, h_orig = img_info["width"], img_info["height"]
        src_path = os.path.join(img_src_dir, fname)
        if not os.path.exists(src_path):
            continue

        dst_img_path = os.path.join(images_dst, fname)
        if not os.path.exists(dst_img_path):
            shutil.copy2(src_path, dst_img_path)

        label_path = os.path.join(labels_dst, Path(fname).stem + ".txt")
        yolo_lines = []
        for ann in anns_by_img.get(img_id, []):
            cat_id = ann["category_id"]
            if cat_id not in cat_to_idx:
                continue  
            
            yolo_cls = cat_to_idx[cat_id] - 1  
            seg = ann.get("segmentation")
            if not seg:
                continue

            if isinstance(seg, list) and len(seg) > 0 and isinstance(seg[0], list):
                poly = seg[0]
                if len(poly) >= 6:
                    norm_poly = [f"{coord / w_orig:.6f}" if i % 2 == 0 else f"{coord / h_orig:.6f}" for i, coord in enumerate(poly)]
                    yolo_lines.append(f"{yolo_cls} " + " ".join(norm_poly))
            elif isinstance(seg, dict):
                mask = segmentation_to_binary_mask(seg, h_orig, w_orig)
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    if len(cnt) >= 3:
                        flattened = cnt.flatten()
                        norm_poly = [f"{coord / w_orig:.6f}" if i % 2 == 0 else f"{coord / h_orig:.6f}" for i, coord in enumerate(flattened)]
                        yolo_lines.append(f"{yolo_cls} " + " ".join(norm_poly))

        with open(label_path, "w") as lf:
            lf.write("\n".join(yolo_lines))

print("[Pipeline] Converting COCO files to YOLO-Seg format in /kaggle/working/yolo_dataset...")
for s_name, i_dir, a_file in [("train", cfg.train_dir, cfg.train_ann),
                              ("valid", cfg.val_dir, cfg.val_ann),
                              ("test", cfg.test_dir, cfg.test_ann)]:
    convert_coco_to_yolo(a_file, i_dir, os.path.join(cfg.yolo_export_dir, s_name), cfg.category_id_to_label)

yolo_yaml_path = os.path.join(cfg.yolo_export_dir, "data.yaml")
with open(yolo_yaml_path, "w") as yf:
    import yaml
    yaml.dump({
        "path": os.path.abspath(cfg.yolo_export_dir),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": cfg.num_classes - 1,
        "names": cfg.class_names[1:]
    }, yf, default_flow_style=False)

# ---------------------------------------------------------------------------
# 2.3 CLAHE and PyTorch Dataloaders with Copy-Paste
# ---------------------------------------------------------------------------
def apply_clahe_lab(image_bgr: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2BGR)

class MinorityInstanceBank:
    def __init__(self, images_dir: str, ann_path: str, cat_to_label: Dict[int, int], max_cache: int = 250):
        self.bank = {label: [] for label in cat_to_label.values()}
        if not os.path.exists(ann_path):
            return
        with open(ann_path, "r") as f:
            coco = json.load(f)
        images = {img["id"]: img for img in coco.get("images", [])}

        for ann in coco.get("annotations", []):
            label = cat_to_label.get(ann["category_id"])
            if label is None or len(self.bank[label]) >= max_cache:
                continue
            img_info = images.get(ann["image_id"])
            if not img_info:
                continue
            img_path = os.path.join(images_dir, img_info["file_name"])
            if not os.path.exists(img_path):
                continue
            image = cv2.imread(img_path)
            if image is None:
                continue
            h, w = image.shape[:2]
            seg = ann.get("segmentation")
            if not seg:
                continue
            mask = segmentation_to_binary_mask(seg, h, w)
            if mask.sum() == 0:
                continue
            x, y, bw, bh = cv2.boundingRect(mask)
            if bw < 5 or bh < 5:
                continue
            self.bank[label].append((image[y:y+bh, x:x+bw].copy(), mask[y:y+bh, x:x+bw].copy()))

    def paste(self, base_img: np.ndarray, base_mask: np.ndarray, n_paste: int = 3) -> Tuple[np.ndarray, np.ndarray]:
        h, w = base_mask.shape
        available_labels = [k for k, v in self.bank.items() if len(v) > 0]
        if not available_labels:
            return base_img, base_mask

        for _ in range(n_paste):
            lbl = random.choice(available_labels)
            crop_img, crop_mask = random.choice(self.bank[lbl])
            ch, cw = crop_mask.shape
            
            min_y = int(h * 0.35)
            max_y = h - ch
            
            # SAFEGUARD: Skip pasting if the crop is physically too tall to fit in the lower 65% of the frame
            if min_y >= max_y or cw >= w:
                continue
                
            y0 = random.randint(min_y, max_y)
            x0 = random.randint(0, w - cw)
            paste_idx = crop_mask.astype(bool)

            roi_img = base_img[y0:y0+ch, x0:x0+cw]
            roi_mask = base_mask[y0:y0+ch, x0:x0+cw]
            roi_img[paste_idx] = crop_img[paste_idx]
            roi_mask[paste_idx] = lbl
            base_img[y0:y0+ch, x0:x0+cw] = roi_img
            base_mask[y0:y0+ch, x0:x0+cw] = roi_mask
        return base_img, base_mask

class SemanticDataset(Dataset):
    def __init__(self, img_dir: str, ann_path: str, cfg, is_train: bool = True):
        self.img_dir = img_dir
        self.cfg = cfg
        self.is_train = is_train
        self.samples = []
        if os.path.exists(ann_path):
            with open(ann_path, "r") as f:
                coco = json.load(f)
            imgs = {img["id"]: img for img in coco.get("images", [])}
            anns = {}
            for a in coco.get("annotations", []):
                anns.setdefault(a["image_id"], []).append(a)
            for i_id, info in imgs.items():
                self.samples.append((info, anns.get(i_id, [])))
        
        self.copy_paste_bank = MinorityInstanceBank(img_dir, ann_path, cfg.category_id_to_label) if is_train else None
        
        self.transforms = A.Compose([
            A.Resize(cfg.img_size, cfg.img_size),
            A.HorizontalFlip(p=0.5 if is_train else 0.0),
            A.RandomBrightnessContrast(p=0.3 if is_train else 0.0),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        info, anns = self.samples[idx]
        img_path = os.path.join(self.img_dir, info["file_name"])
        w_orig, h_orig = info["width"], info["height"]
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((self.cfg.img_size, self.cfg.img_size, 3), dtype=np.uint8)
        
        img = apply_clahe_lab(img)
        mask = np.zeros((h_orig, w_orig), dtype=np.uint8)
        for ann in sorted(anns, key=lambda a: a.get("area", 0), reverse=True):
            lbl = self.cfg.category_id_to_label.get(ann["category_id"], 0)
            seg = ann.get("segmentation")
            if seg and lbl > 0:
                b_mask = segmentation_to_binary_mask(seg, h_orig, w_orig)
                mask[b_mask.astype(bool)] = lbl

        if self.is_train and self.copy_paste_bank and random.random() < 0.5:
            img, mask = self.copy_paste_bank.paste(img, mask)

        augmented = self.transforms(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB), mask=mask)
        return augmented["image"], augmented["mask"].long()

train_ds = SemanticDataset(cfg.train_dir, cfg.train_ann, cfg, is_train=True)
val_ds = SemanticDataset(cfg.val_dir, cfg.val_ann, cfg, is_train=False)

print(f"\n[Dataset Diagnostics] Found {len(train_ds)} Training Images | Found {len(val_ds)} Validation Images")

if len(train_ds) == 0:
    print("❌ CRITICAL ERROR: PyTorch found 0 training images.")
else:
    drop_l = len(train_ds) > cfg.batch_size
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, 
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory, drop_last=drop_l)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, 
                            num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    print("✅ PyTorch DataLoaders successfully constructed.")

[Pipeline] Converting COCO files to YOLO-Seg format in /kaggle/working/yolo_dataset...

[Dataset Diagnostics] Found 1919 Training Images | Found 419 Validation Images
✅ PyTorch DataLoaders successfully constructed.


In [4]:
# %% [Cell 3] Architecture Definition: Custom Hybrid (ResNet50 + ASPP + Coordinate Attention)
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from typing import Optional

# ---------------------------------------------------------------------------
# 3.1 Coordinate Attention Block
# ---------------------------------------------------------------------------
class CoordinateAttention(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 32):
        super().__init__()
        mip = max(8, in_channels // reduction)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1 = nn.Conv2d(in_channels, mip, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish(inplace=True)
        self.conv_h = nn.Conv2d(mip, in_channels, kernel_size=1)
        self.conv_w = nn.Conv2d(mip, in_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        n, c, h, w = x.shape
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        return x * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w.permute(0, 1, 3, 2)))

# ---------------------------------------------------------------------------
# 3.2 ASPP Bottleneck Block
# ---------------------------------------------------------------------------
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256, rates=(6, 12, 18)):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(True))
        ])
        for r in rates:
            self.convs.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(True)
            ))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(True)
        )
        self.project = nn.Sequential(
            nn.Conv2d((len(rates) + 2) * out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(True), nn.Dropout(0.3)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[-2:]
        res = [conv(x) for conv in self.convs]
        res.append(F.interpolate(self.pool(x), size=size, mode="bilinear", align_corners=False))
        return self.project(torch.cat(res, dim=1))

# ---------------------------------------------------------------------------
# 3.3 Complete Hybrid ASPP+CA Segmentation Architecture
# ---------------------------------------------------------------------------
class DecoderBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, use_ca: bool = True):
        super().__init__()
        self.ca = CoordinateAttention(skip_ch) if (skip_ch > 0 and use_ca) else None
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor, skip: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            if self.ca is not None:
                skip = self.ca(skip)
            if skip.shape[-2:] != x.shape[-2:]:
                skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ProposedHybridASPPCA(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        # features_only=True returns multi-scale feature maps from the ResNet50 backbone
        self.encoder = timm.create_model("resnet50", pretrained=pretrained, features_only=True, out_indices=(0, 1, 2, 3, 4))
        chs = self.encoder.feature_info.channels()
        
        self.aspp = ASPP(chs[4], out_channels=256)
        
        # Decoder blocks with Coordinate Attention on the skip connections
        self.b0 = DecoderBlock(256, chs[3], 256, use_ca=True)
        self.b1 = DecoderBlock(256, chs[2], 128, use_ca=True)
        self.b2 = DecoderBlock(128, chs[1], 64, use_ca=True)
        self.b3 = DecoderBlock(64, chs[0], 32, use_ca=True)
        
        self.final = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1, bias=False), 
            nn.BatchNorm2d(32), 
            nn.ReLU(True),
            nn.Conv2d(32, num_classes, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        inp_size = x.shape[-2:]
        f = self.encoder(x)
        
        d = self.b0(self.aspp(f[4]), f[3])
        d = self.b1(d, f[2])
        d = self.b2(d, f[1])
        d = self.b3(d, f[0])
        
        d = F.interpolate(d, size=inp_size, mode="bilinear", align_corners=False)
        return self.final(d)
        
print("✅ Cell 3 Executed: Hybrid Architecture (ResNet50 + ASPP + CA) successfully loaded into memory.")

✅ Cell 3 Executed: Hybrid Architecture (ResNet50 + ASPP + CA) successfully loaded into memory.


In [ ]:
# %% [Cell 4 - Part 1] Training Engine: Proposed Hybrid ASPP+CA Model
import gc
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.amp

# Force override configuration parameters to ensure correct convergence targets
cfg.epochs = 50
cfg.early_stop_patience = 6

# Clean VRAM prior to heavy training initialization
torch.cuda.empty_cache()
gc.collect()

# ---------------------------------------------------------------------------
# 1.1 SotaComboLoss (Focal Tversky + Lovász-Softmax)
# ---------------------------------------------------------------------------
class FocalTverskyLoss(nn.Module):
    def __init__(self, num_classes: int, alpha: float = 0.7, beta: float = 0.3, gamma: float = 0.75):
        super().__init__()
        self.num_classes = num_classes
        self.alpha, self.beta, self.gamma = alpha, beta, gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        targets_1h = F.one_hot(targets.clamp(min=0), self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        tp = torch.sum(probs * targets_1h, dim=dims)
        fn = torch.sum((1 - probs) * targets_1h, dim=dims)
        fp = torch.sum(probs * (1 - targets_1h), dim=dims)
        ti = (tp + 1e-6) / (tp + self.alpha * fn + self.beta * fp + 1e-6)
        return torch.pow(1.0 - ti, self.gamma).mean()

def _lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

class LovaszSoftmaxLoss(nn.Module):
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        c = probs.shape[1]
        probs_flat = probs.permute(0, 2, 3, 1).reshape(-1, c)
        targets_flat = targets.reshape(-1)
        losses = []
        for cl in range(c):
            fg = (targets_flat == cl).float()
            if fg.sum() == 0:
                continue
            errors = (fg - probs_flat[:, cl]).abs()
            errors_sorted, perm = torch.sort(errors, 0, descending=True)
            grad = _lovasz_grad(fg[perm])
            losses.append(torch.dot(errors_sorted, grad))
        return torch.stack(losses).mean() if losses else probs.sum() * 0.0

class SotaComboLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ft = FocalTverskyLoss(cfg.num_classes)
        self.lv = LovaszSoftmaxLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return 0.5 * self.ft(logits, targets) + 0.5 * self.lv(logits, targets)

# ---------------------------------------------------------------------------
# 1.2 Train Hybrid ASPP+CA Loop
# ---------------------------------------------------------------------------
print("\n[Phase 1] Training Proposed Hybrid ASPP+CA...")
hybrid_model = ProposedHybridASPPCA(cfg.num_classes, pretrained=True).to(DEVICE)
if torch.cuda.device_count() > 1:
    hybrid_model = nn.DataParallel(hybrid_model)

raw_model = hybrid_model.module if isinstance(hybrid_model, nn.DataParallel) else hybrid_model

optimizer = torch.optim.AdamW([
    {"params": raw_model.encoder.parameters(), "lr": cfg.lr_backbone, "weight_decay": 1e-2},
    {"params": raw_model.aspp.parameters(), "lr": cfg.lr_aspp_ca, "weight_decay": 1e-3},
    {"params": [p for p in raw_model.parameters() if id(p) not in set(map(id, list(raw_model.encoder.parameters()) + list(raw_model.aspp.parameters())))],
     "lr": cfg.lr_decoder, "weight_decay": 1e-4}
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)
criterion = SotaComboLoss(cfg)

scaler = torch.amp.GradScaler('cuda', enabled=cfg.mixed_precision)

best_hybrid_path = os.path.join(cfg.checkpoint_dir, "best_hybrid.pt")
best_miou = 0.0
epochs_without_precision_improvement = 0

for ep in range(1, cfg.epochs + 1):
    t0 = time.time()
    
    # --- Training Phase ---
    hybrid_model.train()
    running_loss = 0.0
    train_hist = torch.zeros(cfg.num_classes, cfg.num_classes)
    
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda', enabled=cfg.mixed_precision):
            logits = hybrid_model(imgs)
            loss = criterion(logits, masks)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
        
        with torch.no_grad():
            preds = torch.argmax(logits, dim=1).cpu()
            m_valid = (masks.cpu() >= 0) & (masks.cpu() < cfg.num_classes)
            train_hist += torch.bincount(cfg.num_classes * masks.cpu()[m_valid] + preds[m_valid], minlength=cfg.num_classes**2).reshape(cfg.num_classes, cfg.num_classes)
            
    scheduler.step()
    
    train_inter = torch.diag(train_hist)
    train_union = train_hist.sum(1) + train_hist.sum(0) - train_inter
    train_miou = (train_inter[1:] / train_union[1:].clamp(min=1e-6)).mean().item()
    train_epoch_loss = running_loss / len(train_loader.dataset)
    
    # --- Validation Phase ---
    hybrid_model.eval()
    val_running_loss = 0.0
    hist = torch.zeros(cfg.num_classes, cfg.num_classes)
    
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=cfg.mixed_precision):
                logits = hybrid_model(imgs)
                v_loss = criterion(logits, masks)
            
            val_running_loss += v_loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1).cpu()
            m_valid = (masks.cpu() >= 0) & (masks.cpu() < cfg.num_classes)
            hist += torch.bincount(cfg.num_classes * masks.cpu()[m_valid] + preds[m_valid], minlength=cfg.num_classes**2).reshape(cfg.num_classes, cfg.num_classes)
    
    inter = torch.diag(hist)
    union = hist.sum(1) + hist.sum(0) - inter
    val_miou = (inter[1:] / union[1:].clamp(min=1e-6)).mean().item()
    val_epoch_loss = val_running_loss / len(val_loader.dataset)
    
    ep_time = time.time() - t0
    
    print(f"  -> [Hybrid] Epoch {ep:02d}/{cfg.epochs:02d} | Train Loss: {train_epoch_loss:.4f} | Train mIoU: {train_miou:.4f} | Val Loss: {val_epoch_loss:.4f} | Val mIoU: {val_miou:.4f} | Time: {ep_time:.1f}s")
    
    # --- Early Stopping & Model Save Logic ---
    if val_miou > best_miou:
        best_miou = val_miou
        torch.save(raw_model.state_dict(), best_hybrid_path)
        epochs_without_precision_improvement = 0
        print(f"     🌟 New best model saved! (Val mIoU: {best_miou:.4f})")
    else:
        epochs_without_precision_improvement += 1
        print(f"     ⚠️ No improvement for {epochs_without_precision_improvement} epoch(s).")
        if epochs_without_precision_improvement >= cfg.early_stop_patience:
            print(f"🛑 Early stopping triggered after {ep} epochs! Training halted.")
            break

print(f"✅ Saved Best Hybrid Model Checkpoint to: {best_hybrid_path}")

# Thorough cleanup of Phase 1 to free RAM/VRAM
del hybrid_model, raw_model, optimizer, scheduler, criterion, scaler
torch.cuda.empty_cache()
gc.collect()


[Phase 1] Training Proposed Hybrid ASPP+CA...


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

  -> [Hybrid] Epoch 01/50 | Train Loss: 0.7802 | Train mIoU: 0.1776 | Val Loss: 0.6784 | Val mIoU: 0.3745 | Time: 210.9s
     🌟 New best model saved! (Val mIoU: 0.3745)


In [ ]:
# %% [Cell 4 - Part 2] Standalone Training Engine: YOLO11-Seg Instance Model
import gc
import os
import torch
import shutil
from ultralytics import YOLO

# Force override configuration parameters for standalone execution
cfg.yolo_epochs = 50
cfg.early_stop_patience = 6

# Clean VRAM completely before spinning up the YOLO instance engine
torch.cuda.empty_cache()
gc.collect()

print("\n[Phase 2] Training Standalone YOLO11-Seg Instance Model...")
yolo_yaml_path = os.path.join(cfg.yolo_export_dir, "data.yaml")

yolo_model = YOLO(cfg.yolo_model_variant)
yolo_model.train(
    data=yolo_yaml_path,
    epochs=cfg.yolo_epochs,
    imgsz=cfg.img_size,
    batch=cfg.batch_size,
    seed=SEED,
    device=0 if torch.cuda.is_available() else "cpu",
    project=cfg.checkpoint_dir,
    name="yolo11_run",
    exist_ok=True,
    verbose=True,
    patience=cfg.early_stop_patience
)

best_yolo_path = os.path.join(cfg.checkpoint_dir, "best_yolo.pt")
yolo_run_weights = os.path.join(cfg.checkpoint_dir, "yolo11_run", "weights", "best.pt")

if os.path.exists(yolo_run_weights):
    shutil.copy2(yolo_run_weights, best_yolo_path)
    print(f"✅ Saved Best YOLO11 Model Checkpoint to: {best_yolo_path}")
else:
    print(f"⚠️ Warning: YOLO training output not found at {yolo_run_weights}")

# Final system memory wipe
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# %% [Cell 5 (Optional Insert)] Comprehensive Multi-Model Training Curves & Info Report
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------------
# 1. Parse YOLO11-Seg Results CSV with Robust Column Matching
# ---------------------------------------------------------------------------
yolo_results_csv = os.path.join(cfg.checkpoint_dir, "yolo11_run", "results.csv")
yolo_epochs_data = None

def get_matching_col(df: pd.DataFrame, patterns: list) -> str | None:
    for pattern in patterns:
        for col in df.columns:
            if re.search(pattern, col, re.IGNORECASE):
                return col
    return None

yolo_box_loss_col = None
yolo_mask_map50_col = None
yolo_box_map50_col = None

if os.path.exists(yolo_results_csv):
    yolo_df = pd.read_csv(yolo_results_csv)
    yolo_df.columns = [c.strip() for c in yolo_df.columns]
    yolo_epochs_data = yolo_df
    
    yolo_box_loss_col = get_matching_col(yolo_df, [r"train/box_loss", r"box_loss"])
    yolo_mask_map50_col = get_matching_col(yolo_df, [r"metrics/mAP50\(M\)", r"mAP50.*mask", r"mAP_0.5\(M\)"])
    yolo_box_map50_col = get_matching_col(yolo_df, [r"metrics/mAP50\(B\)", r"mAP50.*box", r"mAP_0.5\(B\)"])
    
    print(f"📊 Successfully loaded YOLO11 log from: {yolo_results_csv} ({len(yolo_df)} epochs recorded)")
else:
    print(f"⚠️ Warning: YOLO results file not found at: {yolo_results_csv}")

# Retrieve Hybrid history from memory
hybrid_hist = globals().get('hist_hybrid', globals().get('history', None))

# ---------------------------------------------------------------------------
# 2. Comparative Training Visualization
# ---------------------------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(18, 6), dpi=120)

# --- Subplot 1: Loss Progression ---
hybrid_epochs = range(1, len(hybrid_hist["train_loss"]) + 1) if (hybrid_hist and "train_loss" in hybrid_hist) else []

if len(hybrid_epochs) > 0:
    axes[0].plot(hybrid_epochs, hybrid_hist["train_loss"], label="Hybrid ASPP+CA Train Loss", color="#1f77b4", linewidth=2.2)
    if "val_loss" in hybrid_hist and len(hybrid_hist["val_loss"]) == len(hybrid_epochs):
        axes[0].plot(hybrid_epochs, hybrid_hist["val_loss"], label="Hybrid ASPP+CA Val Loss", color="#1f77b4", linestyle=":", linewidth=1.8)
    axes[0].set_ylabel("Hybrid Combined Loss", color="#1f77b4", fontsize=12)
    axes[0].tick_params(axis='y', labelcolor="#1f77b4")

axes[0].set_title("Loss Convergence Comparison", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Epochs", fontsize=11)
axes[0].grid(True, linestyle="--", alpha=0.5)

if yolo_epochs_data is not None and yolo_box_loss_col:
    ax0_twin = axes[0].twinx()
    yolo_x = yolo_epochs_data["epoch"] if "epoch" in yolo_epochs_data.columns else range(1, len(yolo_epochs_data) + 1)
    ax0_twin.plot(yolo_x, yolo_epochs_data[yolo_box_loss_col], label="YOLO11 Box Loss", color="#ff7f0e", linestyle="--", linewidth=2.0)
    ax0_twin.set_ylabel("YOLO11 Loss Value", color="#ff7f0e", fontsize=12)
    ax0_twin.tick_params(axis='y', labelcolor="#ff7f0e")
    ax0_twin.grid(False)

# Combine legends for Subplot 1
lines_0 = axes[0].get_lines() + (ax0_twin.get_lines() if 'ax0_twin' in locals() else [])
labels_0 = [l.get_label() for l in lines_0]
axes[0].legend(lines_0, labels_0, loc="upper right", frameon=True)

# --- Subplot 2: Metric Evaluation (mIoU vs mAP50) ---
if len(hybrid_epochs) > 0 and "val_miou" in hybrid_hist:
    axes[1].plot(hybrid_epochs, hybrid_hist["val_miou"], label="Hybrid Val mIoU", color="#2ca02c", linewidth=2.2)
    axes[1].set_ylabel("Semantic mIoU Score", color="#2ca02c", fontsize=12)
    axes[1].tick_params(axis='y', labelcolor="#2ca02c")

axes[1].set_title("Accuracy & Detection Progression", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Epochs", fontsize=11)
axes[1].grid(True, linestyle="--", alpha=0.5)

if yolo_epochs_data is not None:
    ax1_twin = axes[1].twinx()
    yolo_x = yolo_epochs_data["epoch"] if "epoch" in yolo_epochs_data.columns else range(1, len(yolo_epochs_data) + 1)
    
    if yolo_mask_map50_col:
        ax1_twin.plot(yolo_x, yolo_epochs_data[yolo_mask_map50_col], label="YOLO11 Mask mAP50", color="#d62728", linestyle="-.", linewidth=2.0)
    if yolo_box_map50_col:
        ax1_twin.plot(yolo_x, yolo_epochs_data[yolo_box_map50_col], label="YOLO11 Box mAP50", color="#9467bd", linestyle=":", linewidth=1.8)
        
    ax1_twin.set_ylabel("YOLO Detection mAP50", color="#d62728", fontsize=12)
    ax1_twin.tick_params(axis='y', labelcolor="#d62728")
    ax1_twin.grid(False)

# Combine legends for Subplot 2
lines_1 = axes[1].get_lines() + (ax1_twin.get_lines() if 'ax1_twin' in locals() else [])
labels_1 = [l.get_label() for l in lines_1]
axes[1].legend(lines_1, labels_1, loc="lower right", frameon=True)

plt.suptitle("Dual-Model Performance Analysis: Hybrid ASPP+CA vs. YOLO11-Seg", fontsize=15, y=1.03, fontweight='bold')
plt.tight_layout()

curves_save_path = os.path.join(cfg.results_dir, "comparative_training_curves.png")
plt.savefig(curves_save_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"📈 Comparative training plot successfully generated -> {curves_save_path}")

# ---------------------------------------------------------------------------
# 3. Dual-Model Benchmark Summary Table & Export
# ---------------------------------------------------------------------------
summary_metrics = {}

if hybrid_hist and "val_miou" in hybrid_hist and len(hybrid_hist["val_miou"]) > 0:
    summary_metrics["Hybrid ASPP+CA Peak Val mIoU"] = [round(max(hybrid_hist["val_miou"]), 4)]
    summary_metrics["Hybrid Final Train Loss"] = [round(hybrid_hist["train_loss"][-1], 4)]
    if "val_loss" in hybrid_hist and len(hybrid_hist["val_loss"]) > 0:
        summary_metrics["Hybrid Best Val Loss"] = [round(min(hybrid_hist["val_loss"]), 4)]

if yolo_epochs_data is not None:
    if yolo_mask_map50_col:
        summary_metrics["YOLO11-Seg Peak Mask mAP50"] = [round(yolo_epochs_data[yolo_mask_map50_col].max(), 4)]
    if yolo_box_map50_col:
        summary_metrics["YOLO11-Seg Peak Box mAP50"] = [round(yolo_epochs_data[yolo_box_map50_col].max(), 4)]
    if yolo_box_loss_col:
        summary_metrics["YOLO11 Lowest Box Loss"] = [round(yolo_epochs_data[yolo_box_loss_col].min(), 4)]

summary_df = pd.DataFrame(summary_metrics).T
summary_df.columns = ["Score"]
summary_csv_path = os.path.join(cfg.results_dir, "dual_model_performance_summary.csv")
summary_df.to_csv(summary_csv_path)

print("\n" + "="*80)
print("🎓 DUAL-MODEL BENCHMARK SUMMARY (Dhaka Dense Traffic Dataset)")
print("="*80)
for metric_name, val in summary_metrics.items():
    print(f"  • {metric_name:<36} : {val[0]:.4f}")
print("="*80)
print(f"📝 Numerical metrics report exported to: {summary_csv_path}\n")

In [ ]:
# %% [Cell 5 (Updated with VRAM Acceleration & Latency Profiling)] Master Ensemble Multi-Sample Qualitative Visualization
import os
import cv2
import time
import random
import numpy as np
from pathlib import Path
from typing import Dict, Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from ultralytics import YOLO

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------------
# 5.1 Merged Master Ensemble Segmenter (Zero-PCIe Bottleneck, Pure VRAM Fusion)
# ---------------------------------------------------------------------------
if 'MergedMasterSegmenter' not in globals():
    class MergedMasterSegmenter:
        def __init__(self, hybrid_weights_path: str, yolo_weights_path: str, cfg):
            self.cfg = cfg
            self.device = getattr(cfg, 'device', torch.device("cuda" if torch.cuda.is_available() else "cpu"))
            
            # Load custom Hybrid Semantic Model (ASPP + Coordinate Attention)
            self.hybrid = ProposedHybridASPPCA(cfg.num_classes, pretrained=False).to(self.device)
            if os.path.exists(hybrid_weights_path):
                self.hybrid.load_state_dict(torch.load(hybrid_weights_path, map_location=self.device))
                print(f"✅ Loaded Hybrid ASPP+CA weights from: {hybrid_weights_path}")
            else:
                print(f"⚠️ Warning: Hybrid weights not found at: {hybrid_weights_path}")
            self.hybrid.eval()

            # Load YOLO11 Instance Segmentation Model
            if os.path.exists(yolo_weights_path):
                self.yolo = YOLO(yolo_weights_path)
                print(f"✅ Loaded YOLO11-Seg weights from: {yolo_weights_path}")
            else:
                print(f"⚠️ Warning: YOLO weights not found at: {yolo_weights_path}. Loading base architecture.")
                self.yolo = YOLO(cfg.yolo_model_variant)

            # Preprocessing pipeline
            self.transform = A.Compose([
                A.Resize(cfg.img_size, cfg.img_size),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2()
            ])
            
            # Warm-up CUDA kernels for low-latency cold start
            self._warmup()

        def _warmup(self):
            if torch.cuda.is_available():
                dummy_bgr = np.zeros((self.cfg.img_size, self.cfg.img_size, 3), dtype=np.uint8)
                _ = self.predict_fused(dummy_bgr)
                torch.cuda.synchronize()

        @torch.no_grad()
        def predict_fused(self, image_bgr: np.ndarray, alpha: float = 0.55, conf_thresh: float = 0.35) -> Tuple[np.ndarray, Dict[str, int], List[Dict], float]:
            """
            Fused Real-Time Predictor:
            - Semantic ASPP+CA provides dense contextual road boundaries.
            - YOLO11-Seg provides sharp instance boundaries & discrete object counts.
            - All tensor interpolations, maximum overlays, and argmax are kept on the GPU.
            """
            t_start = time.perf_counter()
            h_orig, w_orig = image_bgr.shape[:2]

            # 1. CLAHE Light Normalization (Optimized for Dhaka lighting conditions)
            lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clahe_img = cv2.cvtColor(cv2.merge((cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l), a, b)), cv2.COLOR_LAB2BGR)
            rgb = cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB)

            # Push image tensor to VRAM
            tensor = self.transform(image=rgb)["image"].unsqueeze(0).to(self.device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=self.cfg.mixed_precision):
                # 2. Semantic Stream Inference & Native GPU Resizing
                logits = self.hybrid(tensor)
                probs_semantic = F.softmax(logits, dim=1)
                probs_semantic_resized = F.interpolate(
                    probs_semantic, size=(h_orig, w_orig), mode='bilinear', align_corners=False
                ).squeeze(0)  # Shape: [num_classes, H, W]

                # 3. YOLO Instance Segmentation Stream
                yolo_res = self.yolo.predict(
                    source=image_bgr,
                    conf=conf_thresh,
                    imgsz=self.cfg.img_size,
                    device=self.device,
                    verbose=False
                )[0]

                # Initialize Instance Probability Tensor directly on GPU
                probs_instance = torch.zeros(
                    (self.cfg.num_classes, h_orig, w_orig),
                    dtype=probs_semantic_resized.dtype,
                    device=self.device
                )
                probs_instance[0, :, :] = 0.5  # Neutral background baseline

                counts = {}
                detections = []
                names = yolo_res.names

                if yolo_res.boxes is not None and yolo_res.masks is not None and len(yolo_res.boxes) > 0:
                    # Batch-resize all predicted masks concurrently on the GPU
                    masks_gpu = yolo_res.masks.data.unsqueeze(1).float()  # Shape: [N, 1, H_mask, W_mask]
                    masks_resized = F.interpolate(
                        masks_gpu, size=(h_orig, w_orig), mode='nearest'
                    ).squeeze(1).to(probs_semantic_resized.dtype)

                    for i, box in enumerate(yolo_res.boxes):
                        cls_id = int(box.cls[0].item())
                        cls_name = names.get(cls_id, f"class_{cls_id}")
                        conf = float(box.conf[0].item())

                        counts[cls_name] = counts.get(cls_name, 0) + 1
                        detections.append({
                            "class": cls_name,
                            "confidence": round(conf, 3),
                            "bbox": [round(c, 1) for c in box.xyxy[0].tolist()]
                        })

                        # Map YOLO class index to semantic index (Index 0 = background)
                        target_cls = cls_id + 1
                        if target_cls < self.cfg.num_classes:
                            conf_tensor = torch.tensor(conf, dtype=probs_semantic_resized.dtype, device=self.device)
                            probs_instance[target_cls] = torch.maximum(
                                probs_instance[target_cls],
                                masks_resized[i] * conf_tensor
                            )

                # 4. In-VRAM Weighted Fusion & Class Extraction
                fused_probs = (alpha * probs_semantic_resized) + ((1.0 - alpha) * probs_instance)
                final_mask_gpu = torch.argmax(fused_probs, dim=0).to(torch.uint8)

                # Single PCIe transfer: send only the final 2D mask to CPU RAM
                final_mask = final_mask_gpu.cpu().numpy()

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            latency_ms = (time.perf_counter() - t_start) * 1000.0

            return final_mask, counts, detections, latency_ms

    best_hybrid_path = os.path.join(cfg.checkpoint_dir, "best_hybrid.pt")
    best_yolo_path = os.path.join(cfg.checkpoint_dir, "best_yolo.pt")
    master_segmenter = MergedMasterSegmenter(best_hybrid_path, best_yolo_path, cfg)

# ---------------------------------------------------------------------------
# 5.2 Multi-Sample Qualitative Evaluation Grid with Class Legend & Latency
# ---------------------------------------------------------------------------
test_imgs = (
    list(Path(cfg.val_dir).glob("*.jpg")) + 
    list(Path(cfg.val_dir).glob("*.jpeg")) + 
    list(Path(cfg.val_dir).glob("*.png")) +
    list(Path(cfg.val_dir).glob("*.JPG"))
)

if len(test_imgs) >= 2:
    sample_paths = random.sample(test_imgs, 2)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=120)

    # Generate HSV Colormap for discrete classes
    colors = np.zeros((cfg.num_classes, 3), dtype=np.uint8)
    for i in range(1, cfg.num_classes):
        rgb = mcolors.hsv_to_rgb([(i - 1) / max(1, (cfg.num_classes - 1)), 0.85, 0.95])
        colors[i] = (np.array(rgb) * 255).astype(np.uint8)

    for idx, img_path in enumerate(sample_paths):
        bgr = cv2.imread(str(img_path))
        if bgr is None:
            continue
        rgb_input = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        # Execute GPU-accelerated fusion
        f_mask, v_counts, v_dets, lat_ms = master_segmenter.predict_fused(bgr, alpha=0.55, conf_thresh=0.35)
        fps = 1000.0 / max(1e-3, lat_ms)

        color_mask = colors[f_mask]
        overlay = cv2.addWeighted(rgb_input, 0.6, color_mask, 0.4, 0)

        # Top Row: Original Traffic Scenes
        axes[0, idx].imshow(rgb_input)
        axes[0, idx].set_title(f"Validation Frame {idx+1}: {img_path.name}", fontsize=11, fontweight='bold')
        axes[0, idx].axis("off")

        # Bottom Row: Real-Time Fused Segmentations + Detections
        inv_str = ", ".join([f"{k}: {v}" for k, v in v_counts.items()]) if v_counts else "None"
        axes[1, idx].imshow(overlay)
        axes[1, idx].set_title(
            f"Master Fused Output {idx+1} | Latency: {lat_ms:.1f}ms (~{fps:.1f} FPS)\nObjects Count: [{inv_str}]",
            fontsize=10,
            fontweight='bold'
        )
        axes[1, idx].axis("off")

    # Add Academic Color Legend across all active classes
    legend_patches = [
        mpatches.Patch(color=colors[i] / 255.0, label=cfg.class_names[i]) 
        for i in range(1, cfg.num_classes)
    ]
    fig.legend(
        handles=legend_patches,
        loc="lower center",
        ncol=min(cfg.num_classes - 1, 7),
        bbox_to_anchor=(0.5, -0.02),
        fontsize=10,
        frameon=True
    )

    plt.suptitle("Master Ensemble Qualitative Evaluation: Custom Hybrid ASPP+CA + YOLO11-Seg", fontsize=14, y=0.98, fontweight='bold')
    plt.tight_layout()

    grid_img_path = os.path.join(cfg.results_dir, "master_fused_2x2_random_grid.png")
    plt.savefig(grid_img_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"🖼️ Master Ensemble grid saved successfully -> {grid_img_path}")
else:
    print(f"⚠️ Warning: Found {len(test_imgs)} validation images. Need at least 2 for the 2x2 grid.")

In [ ]:
# %% [Cell 7 (Stunning Thesis Edition)] Vehicle Class Distribution, Imbalance & Novelty Analysis
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ---------------------------------------------------------------------------
# 7.1 Parse COCO Annotations & Compute Distribution Metrics
# ---------------------------------------------------------------------------
def audit_dataset_classes(ann_path: str, class_names: list):
    if not os.path.exists(ann_path):
        return None
    with open(ann_path, "r") as f:
        coco = json.load(f)
        
    cat_id_to_name = {c["id"]: c["name"] for c in coco.get("categories", [])}
    valid_classes = class_names[1:]
    instance_counts = {c: 0 for c in valid_classes}
    pixel_areas = {c: 0.0 for c in valid_classes}
    
    for ann in coco.get("annotations", []):
        cname = cat_id_to_name.get(ann.get("category_id"))
        if cname in instance_counts:
            instance_counts[cname] += 1
            pixel_areas[cname] += float(ann.get("area", 0.0))
            
    df_classes = pd.DataFrame({
        "Vehicle Class": list(instance_counts.keys()),
        "Total Instances": list(instance_counts.values()),
        "Total Pixel Area": list(pixel_areas.values())
    })
    
    total_inst = df_classes["Total Instances"].sum()
    total_pix = df_classes["Total Pixel Area"].sum()
    df_classes["Instance Share (%)"] = (df_classes["Total Instances"] / max(1, total_inst)) * 100
    df_classes["Pixel Area Share (%)"] = (df_classes["Total Pixel Area"] / max(1, total_pix)) * 100
    return df_classes

train_audit_csv = audit_dataset_classes(cfg.train_ann, cfg.class_names)

if train_audit_csv is not None:
    # Set professional thesis styling
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Left Subplot: Instance Counts with custom color palette emphasizing minority classes
    palette_inst = ["#2b5c8f" if x not in ["motorcycle", "leguna"] else "#e76f51" for x in train_audit_csv["Vehicle Class"]]
    sns.barplot(data=train_audit_csv, x="Vehicle Class", y="Total Instances", ax=axes[0], palette=palette_inst)
    axes[0].set_title("Instance-Level Frequency & Class Imbalance", fontsize=13, fontweight='bold', pad=12)
    axes[0].set_ylabel("Total Instances", fontsize=11, fontweight='semibold')
    axes[0].tick_params(axis="x", rotation=25, labelsize=10)
    
    # Right Subplot: Spatial Pixel Share
    palette_pix = ["#2a9d8f" if x not in ["motorcycle", "leguna"] else "#f4a261" for x in train_audit_csv["Vehicle Class"]]
    sns.barplot(data=train_audit_csv, x="Vehicle Class", y="Pixel Area Share (%)", ax=axes[1], palette=palette_pix)
    axes[1].set_title("Pixel-Level Spatial Dominance Share (%)", fontsize=13, fontweight='bold', pad=12)
    axes[1].set_ylabel("Foreground Area Share (%)", fontsize=11, fontweight='semibold')
    axes[1].tick_params(axis="x", rotation=25, labelsize=10)
    
    plt.suptitle("Dhaka Urban Traffic Dataset: Fine-Grained Class Imbalance & Spatial Footprint", fontsize=15, fontweight='bold', y=1.03)
    plt.tight_layout()
    
    chart_path = os.path.join(cfg.results_dir, "vehicle_class_imbalance_analysis.png")
    plt.savefig(chart_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"📊 Stunning class distribution chart saved successfully to: {chart_path}")

    # ---------------------------------------------------------------------------
    # 7.2 Print Academic Summary, Minority Breakdown & Research Novelty Proof
    # ---------------------------------------------------------------------------
    print("\n" + "="*85)
    print("🎓 THESIS SECTION REPORT: CLASS IMBALANCE, MINORITY CLASSES & ARCHITECTURAL NOVELTY")
    print("="*85)
    print(train_audit_csv.to_string(index=False))
    print("-"*85)
    print("🔍 CRITICAL FINDINGS & MINORITY CLASS DYNAMICS:")
    print("• Severe Instance Imbalance Identified: Motorcycles (97 instances / 8.64%) and Legunas")
    print("  (102 instances / 9.08%) constitute the primary minority classes by frequency.")
    print("• The 'Leguna' Paradox: Despite having a low instance count (102), Legunas command the single")
    print("  highest spatial pixel footprint across the dataset at 23.34% total area share, making")
    print("  accurate multi-scale boundary localization critical.")
    print("-" * 85)
    print("💡 OUR RESEARCH NOVELTY & TECHNICAL CONTRIBUTION:")
    print("1. Coordinate Attention (CA) Integration: Traditional architectures fail on minority classes")
    print("   like motorcycles and legunas due to extreme occlusion and scale variance. By embedding CA")
    print("   directly into skip connections, our model encodes horizontal and vertical spatial position")
    print("   into channel attention, recovering fine-grained boundary details.")
    print("2. ASPP Contextual Bottleneck: Resolves scale discrepancies between small two-wheelers and large buses.")
    print("3. SotaComboLoss Optimization: Combines Focal Tversky and Lovász-Softmax to directly optimize")
    print("   for mIoU, preventing background collapse and protecting minority class gradients.")
    print("="*85 + "\n")

In [ ]:
# %% [Cell 8] Real-Time Video Inference & Master Ensemble Counting Pipeline
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from ultralytics import YOLO
import matplotlib.colors as mcolors
from pathlib import Path

# ---------------------------------------------------------------------------
# 8.1 Video Inference Engine
# ---------------------------------------------------------------------------
def run_master_video_inference(input_video_path: str, output_video_path: str, master_segmenter, cfg, max_frames: int = None):
    if not os.path.exists(input_video_path):
        print(f"❌ Error: Input video not found at {input_video_path}")
        return

    cap = cv2.VideoCapture(input_video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # Generate static colormap for classes
    colors = np.zeros((cfg.num_classes, 3), dtype=np.uint8)
    for i in range(1, cfg.num_classes):
        rgb = mcolors.hsv_to_rgb([(i - 1) / max(1, (cfg.num_classes - 1)), 0.85, 0.95])
        colors[i] = (np.array(rgb) * 255).astype(np.uint8)

    print(f"\n🎥 Starting Video Processing: {Path(input_video_path).name}")
    print(f"• Resolution: {width}x{height} @ {fps} FPS | Total Frames: {total_frames}")
    
    frame_idx = 0
    while cap.isOpened():
        ret, frame_bgr = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        if max_frames and frame_idx > max_frames:
            break

        # Run Master Ensemble Prediction
        f_mask, v_counts, v_dets = master_segmenter.predict_fused(frame_bgr, alpha=0.55, conf_thresh=0.35)
        
        # Apply Segmentation Overlay
        color_mask = colors[f_mask]
        overlay_frame = cv2.addWeighted(frame_bgr, 0.6, color_mask, 0.4, 0)
        
        # Draw YOLO Bounding Boxes & Class Names
        for det in v_dets:
            box = det["bbox"]
            cls_name = det["class"]
            conf = det["confidence"]
            x1, y1, x2, y2 = map(int, box)
            
            # Color-coded box based on class
            cv2.rectangle(overlay_frame, (x1, y1), (x2, y2), (0, 255, 128), 2)
            label_text = f"{cls_name}: {conf:.2f}"
            
            # Label background banner for readability
            (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(overlay_frame, (x1, y1 - 20), (x1 + tw + 4, y1), (0, 255, 128), -1)
            cv2.putText(overlay_frame, label_text, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

        # Overlay Live Inventory Count Panel on Top-Left Corner
        panel_y = 35
        cv2.putText(overlay_frame, f"Frame: {frame_idx} | Master Ensemble Live Count:", (20, panel_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        
        for cname, count in v_counts.items():
            panel_y += 25
            cv2.putText(overlay_frame, f" - {cname}: {count}", (30, panel_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2, cv2.LINE_AA)

        out.write(overlay_frame)
        if frame_idx % 30 == 0:
            print(f"  -> Processed Frame {frame_idx}/{total_frames}...")

    cap.release()
    out.release()
    print(f"✅ Master Video Inference Complete! Saved to: {output_video_path}")

# ---------------------------------------------------------------------------
# 8.2 Execution on Test Video
# ---------------------------------------------------------------------------
# Provide your test video path here (e.g., a sample traffic video from your thesis dataset)
input_video = "/kaggle/input/datasets/sadidahmedturjo/trafic-video/gettyimages-2269287495-640_adpp.mp4" # Update with your video path if needed
output_video = os.path.join(cfg.results_dir, "master_ensemble_traffic_output.mp4")

# Fallback check if user needs to upload or specify path
if not os.path.exists(input_video):
    print(f"⚠️ Notice: Default path '{input_video}' not found.")
    print("Please place your test video path into `input_video` variable and re-run.")
else:
    run_master_video_inference(input_video, output_video, master_segmenter, cfg, max_frames=300)


In [ ]:
# %% [Cell 9] Compress & Secure All Thesis Outputs for Download
import os
import zipfile
from IPython.display import FileLink

output_dir = cfg.results_dir
checkpoint_dir = cfg.checkpoint_dir
zip_filename = "/kaggle/working/dhaka_traffic_thesis_assets.zip"

print("📦 Packaging all model checkpoints and visual results...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add checkpoints (best_hybrid.pt, best_yolo.pt)
    for root, dirs, files in os.walk(checkpoint_dir):
        for file in files:
            if file.endswith(".pt"):
                full_path = os.path.join(root, file)
                zipf.write(full_path, arcname=os.path.join("checkpoints", file))
                
    # Add thesis results and plots
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            full_path = os.path.join(root, file)
            zipf.write(full_path, arcname=os.path.join("results", file))

print(f"✅ Successfully created archive: {zip_filename}")

# Generate a direct download link right inside your notebook interface
display(FileLink(zip_filename))